In [ ]:
# import the required libraries
import pandas as pd
import torch
import json

In [ ]:
# read the input data (embedded keywords-title dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the embedded keywords-title dataset is the dataset 
formed by merging the embedded keywords and embedded title dataset
"""
movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/bert-embed/embedded-data/mkg_title_dataset_bert.csv")

In [ ]:
movies_keywords_genres.head(1)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,keywords_embedding,original_title_embedding
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.07,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,[ 2.24766999e-01 1.48856416e-01 -4.26082835e-...,[-3.27251524e-01 -1.14664100e-01 -1.55884609e-...


In [ ]:
movies_keywords_genres.shape[0]

234753

In [ ]:
genre_index_mapping = dict()
txt_file_path = '/content/drive/MyDrive/LLM-CB-Embed/BERT-Embed/genre_mapping.txt'
with open(txt_file_path, 'r') as txt_file:
    genre_index_mapping = json.loads(txt_file.read())
print("Genre Index Mapping: \n", genre_index_mapping)

Genre Index Mapping: 
 {'Crime': 1, 'Western': 2, 'Fantasy': 3, 'Science Fiction': 4, 'Comedy': 5, 'Family': 6, 'Mystery': 7, 'Drama': 8, 'Animation': 9, 'Thriller': 10, 'Documentary': 11, 'Horror': 12, 'Romance': 13, 'TV Movie': 14, 'History': 15, 'War': 16, 'Adventure': 17, 'Music': 18, 'Action': 19}


In [ ]:
# Map genres to numeric indices in the DataFrame and create a new column 'genre_indices'
movies_keywords_genres['genres_indices'] = movies_keywords_genres['genres'].apply(lambda genres: [genre_index_mapping[genre.strip()] for genre in genres.split(',') if pd.notna(genres)])

In [ ]:
movies_keywords_genres.head(5)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,keywords_embedding,original_title_embedding,genres_indices
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,[ 2.24766999e-01 1.48856416e-01 -4.26082835e-...,[-3.27251524e-01 -1.14664100e-01 -1.55884609e-...,"[11, 8]"
1,False,1115,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[ 8.62499252e-02 6.64695725e-02 4.50119734e-...,[-3.33471775e-01 -3.40431213e-01 1.28486112e-...,[8]
2,False,1116,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,[ 2.55626798e-01 3.36861275e-02 -1.86209410e-...,[-1.84306309e-01 7.14582950e-02 -1.47662491e-...,"[8, 16]"
3,False,1123,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,[-1.35598034e-01 -3.07193458e-01 -3.09346825e-...,[ 4.00660694e-01 -1.56783655e-01 -1.28297895e-...,"[19, 8, 10]"
4,False,1124,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,[ 3.90980601e-01 2.90082335e-01 5.26184797e-...,[ 3.64068747e-02 -1.96709305e-01 -4.71620530e-...,"[8, 7, 4]"


In [ ]:
# convert the embedded column array to embedded string and clean the keywords_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_keywords_genres['keywords_embedding_string'] = movies_keywords_genres['keywords_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')

<ipython-input-8-481dfa4f3b59>:1: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  movies_keywords_genres['keywords_embedding_string'] = movies_keywords_genres['keywords_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')


In [ ]:
movies_keywords_genres.loc[0, 'keywords_embedding_string']

' 2.24766999e-01  1.48856416e-01 -4.26082835e-02 -3.14322889e-01  2.65974730e-01  6.07257001e-02  4.41800714e-01  1.39571846e-01 -1.91153675e-01 -9.01168734e-02  1.11876912e-01 -1.91026092e-01  9.28930044e-02  4.00517941e-01 -6.45102799e-01  4.05014932e-01  2.12058440e-01  9.12833773e-03 -1.12280786e-01  1.07610442e-01  8.39885622e-02 -2.40673110e-01 -2.28782520e-02  2.58007646e-01  2.93938667e-01  4.24413607e-02  7.02994689e-03  4.05519485e-01  7.91630298e-02 -7.67677501e-02  5.70432395e-02 -1.53115429e-02  1.35083154e-01 -4.52344492e-02  1.41073078e-01 -1.58745851e-02 -1.72262028e-01 -1.57738924e-01  5.68578094e-02  3.31399381e-01 -1.68964833e-01 -6.03243947e-01 -1.90544814e-01  9.75969881e-02 -4.77220789e-02 -3.24775577e-01 -8.70985836e-02  3.28057408e-01  2.34474182e-01 -1.79024369e-01 -2.94766724e-01  2.09116891e-01 -8.72850195e-02 -2.23440021e-01  2.38965124e-01  6.68386102e-01  2.42221326e-01 -3.08476210e-01 -8.44973102e-02  1.43814087e-02 -6.81748167e-02 -5.05768806e-02  9.5229

In [ ]:
# convert the embedded array to embedded string and clean the original_title_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_keywords_genres['original_title_embedding_string'] = movies_keywords_genres['original_title_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')

<ipython-input-10-c287eab7b5aa>:1: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  movies_keywords_genres['original_title_embedding_string'] = movies_keywords_genres['original_title_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')


In [ ]:
movies_keywords_genres.loc[0, 'original_title_embedding_string']

'-3.27251524e-01 -1.14664100e-01 -1.55884609e-01 -6.56328723e-02  1.96430460e-01 -2.80801088e-01  5.54966748e-01  1.88514903e-01 -9.07844603e-02 -1.35775656e-01  1.19555434e-02 -5.03463745e-01  3.46662581e-01  4.26864505e-01 -1.88873515e-01  5.68338990e-01 -4.93274294e-02 -4.73374069e-01 -1.09931156e-01  2.78008729e-01 -3.63493800e-01  1.14051126e-01  1.89836726e-01  2.82690823e-01  3.47331256e-01  3.43999341e-02 -3.06722708e-02 -7.00706989e-02 -1.26827225e-01 -6.32136986e-02  2.33529747e-01  3.13521773e-02  1.28958002e-01  4.29000586e-01 -8.53547826e-02  2.08193257e-01  1.33735061e-01  7.86781460e-02 -3.20499856e-03  2.25116491e-01 -8.36772323e-01 -3.60116929e-01 -3.48444790e-01 -4.39322330e-02 -3.93483460e-01 -6.48988008e-01 -3.91291916e-01  5.17050743e-01  1.40956859e-03  1.34038121e-01 -1.96864232e-01 -3.32238451e-02 -7.37357065e-02  3.22577208e-01 -1.66661918e-01  6.71784580e-02 -3.02146256e-01 -1.06318653e+00  3.00292283e-01  8.14327002e-02  3.10709029e-01 -2.10846886e-01  4.2419

In [ ]:
movies_keywords_genres.head(2)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,keywords_embedding,original_title_embedding,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.07,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,[ 2.24766999e-01 1.48856416e-01 -4.26082835e-...,[-3.27251524e-01 -1.14664100e-01 -1.55884609e-...,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...
1,False,1115,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.30,26.0,1115.0,friendship berlin germany unsociability loss o...,[ 8.62499252e-02 6.64695725e-02 4.50119734e-...,[-3.33471775e-01 -3.40431213e-01 1.28486112e-...,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...


In [ ]:
# drop the embedded array columns for keywords and original_title
drop_columns = ['keywords_embedding', 'original_title_embedding']
movies_keywords_genres = movies_keywords_genres.drop(columns=drop_columns)
movies_keywords_genres.head(5)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...
1,False,1115,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...
2,False,1116,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",2.55626798e-01 3.36861275e-02 -1.86209410e-0...,-1.84306309e-01 7.14582950e-02 -1.47662491e-0...
3,False,1123,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-1.35598034e-01 -3.07193458e-01 -3.09346825e-0...,4.00660694e-01 -1.56783655e-01 -1.28297895e-0...
4,False,1124,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",3.90980601e-01 2.90082335e-01 5.26184797e-0...,3.64068747e-02 -1.96709305e-01 -4.71620530e-0...


In [ ]:
# remove any NaN rows & columns
nan_count = movies_keywords_genres['keywords_embedding_string'].isnull().sum(); print(f"The column 'keywords_embedding_string' has {nan_count} NaN values.")
nan_count = movies_keywords_genres['original_title_embedding_string'].isnull().sum(); print(f"The column 'original_title_embedding_string' has {nan_count} NaN values.")
nan_count = movies_keywords_genres['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
movies_keywords_genres.dropna(subset=['keywords_embedding_string'], inplace=True)
movies_keywords_genres.dropna(subset=['original_title_embedding_string'], inplace=True)
movies_keywords_genres.dropna(subset=['genres'], inplace=True)

The column 'keywords_embedding_string' has 34 NaN values.
The column 'original_title_embedding_string' has 0 NaN values.
The column 'genres' has 0 NaN values.


In [ ]:
movies_keywords_genres.shape[0]

234719

In [ ]:
# export the cleaned embedded keywords and title dataset to a .csv file
movies_keywords_genres.to_csv('/content/drive/MyDrive/LLM-NEW-2/bert-embed/embedded-data/final_mkg_dataset_bert.csv', index=False)